## Imports & Pfade

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_ROOT = Path("../data")
HARM_DIR = DATA_ROOT / "processed" / "harmonized_250hz"
OUT_DIR  = DATA_ROOT / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert HARM_DIR.exists(), f"Nicht gefunden: {HARM_DIR.resolve()}"
npz_files = sorted(HARM_DIR.glob("*.npz"))

print("NPZ files found:", len(npz_files))
print("Example files:", [p.name for p in npz_files[:5]])


## Segmentierungsparameter (5s / 2.5s / 250 Hz)

In [ ]:
FS = 250
WINDOW_S = 5.0
STEP_S = 2.5

WINDOW_N = int(WINDOW_S * FS)  
STEP_N   = int(STEP_S * FS)    

print("WINDOW_N:", WINDOW_N, "samples")
print("STEP_N  :", STEP_N, "samples")


## Intervalle-Hilfsfunktion

In [ ]:
def is_in_any_interval(t_s: float, intervals_s: np.ndarray) -> bool:
    if intervals_s is None or intervals_s.size == 0:
        return False
    starts = intervals_s[:, 0]
    ends   = intervals_s[:, 1]
    return bool(np.any((t_s >= starts) & (t_s <= ends)))


## Segment-Index für eine Datei bauen

In [ ]:
def build_segment_index_for_file(npz_path: Path):
    d = np.load(npz_path, allow_pickle=True)

    signal = d["signal"]
    fs = int(d["fs"])
    if fs != FS:
        raise ValueError(f"{npz_path.name}: fs={fs} != expected {FS}")

    source = str(d["source"])
    record = str(d["record"])

    mal_intervals = d["malignant_intervals_s"].astype(float)

    noise_intervals = d["noise_intervals_s"].astype(float) if "noise_intervals_s" in d.files else np.empty((0, 2))

    n_samples = signal.shape[0]
    rows = []

    for start_idx in range(0, n_samples - WINDOW_N + 1, STEP_N):
        end_idx = start_idx + WINDOW_N
        mid_idx = start_idx + WINDOW_N // 2

        start_s = start_idx / fs
        end_s   = end_idx / fs
        mid_s   = mid_idx / fs

        if is_in_any_interval(mid_s, noise_intervals):
            y = -1 
        elif is_in_any_interval(mid_s, mal_intervals):
            y = 1
        else:
            y = 0

        rows.append({
            "source": source,
            "record": record,
            "npz_file": npz_path.name,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "start_s": start_s,
            "end_s": end_s,
            "mid_s": mid_s,
            "label": y
        })

    return pd.DataFrame(rows)



## Alle Dateien segmentieren

In [ ]:
dfs = []
for p in npz_files:
    dfs.append(build_segment_index_for_file(p))

seg_index = pd.concat(dfs, ignore_index=True)

print("Total segments:", len(seg_index))
seg_index.head()


## Speichern 

In [ ]:
seg_index_clean = seg_index[seg_index["label"] != -1].copy()
seg_index.to_parquet(OUT_DIR / "segments_5s_2p5s_index_with_noise.parquet", index=False)
seg_index_clean.to_parquet(OUT_DIR / "segments_5s_2p5s_index_clean.parquet", index=False)

print("Saved:", (OUT_DIR / "segments_5s_2p5s_index_with_noise.parquet").resolve())
print("Saved:", (OUT_DIR / "segments_5s_2p5s_index_clean.parquet").resolve())
